Data Pre-processing

In [1]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

c:\Users\nraja\OneDrive\Desktop\llama\llm_engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

In [2]:
username = "SujithVarma2005"
LITE_MODE = True
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='Odyssey K45120BLK Krom Utility/Record Case, Black' category='Musical_Instruments' price=119.95 full='Odyssey  Krom Utility/Record Case, Black\n[\'Protect your rare and hard to find records used for your special 45 rpm vinyl sets or just for safe keeping in our Krom™ series  black utility/record cases. Each has two individual compartments that hold a total of 120 7" vinyl records, 60 in each section, and can also be used as a utility case for cables and other essentials. Features include chrome plated hardware, a removable lid, a fully foam-lined interior and a heavy-duty latch and spring-loaded handle. Individual K45120 cases can be stacked on each other for convenient storage. Also available in silver ||.\']\n[\'Butterfly latch and spring-loaded handle\', \'Padlock latch holes\', \'Holds 120 7" vinyl records, 60 per compartment\', \'Foam-lined interior\', \'Rubber feet and stacking lid\']\n{"Item Weight": "7 pounds", "Product Dimensions": "9.25 x 17.75 x 9.7

20,000 train
1,000 validation
1,000 test
----------------
22,000 items total

In [3]:
print(items[3])

title='Lenovo ThinkPad P50 15.6 inches FHD Laptop, Core i7-6700HQ 2.6GHz, 16GB RAM, 480GB Solid State Drive, Windows 10 Pro 64bit (Renewed)' category='Electronics' price=508.0 full='Lenovo ThinkPad P50 15.6 inches FHD Laptop, Core i7-6700HQ 2.6GHz, 16GB RAM, 480GB Solid State Drive, Windows 10 Pro 64bit (Renewed)\n[\'This pre-owned or refurbished product has been professionally inspected and tested to work and look like new. How a product becomes part of Amazon Renewed, your destination for pre-owned, refurbished products: A customer buys a new product and returns it or trades it in for a newer or different model. That product is inspected and tested to work and look like new by Amazon-qualified suppliers. Then, the product is sold as an Amazon Renewed product on Amazon. If not satisfied with the purchase, renewed products are eligible for replacement or refund under the Amazon Renewed Guarantee.\']\n[\'Product Type: \', \'Item Package Quantity:1\', "Display : 14\' FHD screen", \'Proce

In [4]:
# Give every item an id
for index, item in enumerate(items):
    item.id = index

In [5]:
print(item.id)

21999


In [6]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [7]:
print(items[0].full)

Odyssey  Krom Utility/Record Case, Black
['Protect your rare and hard to find records used for your special 45 rpm vinyl sets or just for safe keeping in our Krom™ series  black utility/record cases. Each has two individual compartments that hold a total of 120 7" vinyl records, 60 in each section, and can also be used as a utility case for cables and other essentials. Features include chrome plated hardware, a removable lid, a fully foam-lined interior and a heavy-duty latch and spring-loaded handle. Individual K45120 cases can be stacked on each other for convenient storage. Also available in silver ||.']
['Butterfly latch and spring-loaded handle', 'Padlock latch holes', 'Holds 120 7" vinyl records, 60 per compartment', 'Foam-lined interior', 'Rubber feet and stacking lid']
{"Item Weight": "7 pounds", "Product Dimensions": "9.25 x 17.75 x 9.75 inches", "Is Discontinued By Manufacturer": "No", "Date First Available": "September 6, 2012", "Color Name": "BLACK", "Material Type": "Rubbe

Now we will do by performing model that is Groq

In [10]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": items[0].full}
]
response = completion(messages=messages, model = "groq/llama-3.1-8b-instant")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Rewritten short precise title: Odyssey Krom Utility/Record Case
Category: Electronics
Brand: Odyssey
Description: A versatile and heavily equipped black record case holding up to 120 7-inch vinyl records and serving as a utility storage for essentials.
Details: Features include foam-lined interior, removable lid, chrome plated hardware, latch and spring-loaded handle for added convenience and protection.

Input tokens: 378
Output tokens: 78
Cost: 0.003 cents


Now we will run ollama

In [11]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

### Product:
Title: Vinyl Record Storage Case
Category: Electronics
Brand: Odyssey
Description: A black, utility-style case with two compartments to store up to 120 7" vinyl records.
Details: Features chrome-plated hardware and a foam-lined interior for secure storage.

Input tokens: 375
Output tokens: 58
Cost: 0.000 cents


Since groq has more valuable content so we will use groq model 

We should make json file

In [12]:
MODEL = "openai/llama-3.1-8b-instant"

In [13]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [14]:
items[0]

<Odyssey K45120BLK Krom Utility/Record Case, Black = $119.95>

In [15]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/llama-3.1-8b-instant", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Odyssey  Krom Utility/Record Case, Black\\n[\'Protect your rare and hard to find records used for your special 45 rpm vinyl sets or just for safe keeping in our Krom\\u2122 series  black utility/record cases. Each has two individual compartments that hold a total of 120 7\\" vinyl records, 60 in each section, and can also be used as a utility case for cables and other essentials. Features include chrome plated hardware, a removable lid, a fully foam-lined interior and a heavy-duty latch and spring-loaded handle. Individual K45120 cases can b

In [16]:
def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [17]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

We are using Groq 

In [18]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

Reads from JSONL file

In [26]:
import json
import time
from litellm import completion

responses = []

with open("jsonl/0_1000.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):

        # Process first 50 requests from the JSONL file
        if i >= 50:
            break

        request = json.loads(line)

        while True:
            try:
                response = completion(
                    model="groq/openai/gpt-oss-safeguard-20b",
                    messages=request["body"]["messages"]
                )

                result = {
                    "custom_id": request["custom_id"],
                    "response": response.choices[0].message.content
                }

                responses.append(result)

                print(f"Completed {i+1}/50")
                time.sleep(3)
                break

            except Exception as e:
                print(f"Error: {e}")
                print("Waiting 10 seconds...")
                time.sleep(10)

print(f"Finished {len(responses)} requests")

Completed 1/50
Completed 2/50
Completed 3/50
Completed 4/50
Completed 5/50
Completed 6/50
Completed 7/50
Completed 8/50
Completed 9/50
Completed 10/50
Completed 11/50
Completed 12/50
Completed 13/50
Completed 14/50
Completed 15/50
Completed 16/50
Completed 17/50
Completed 18/50
Completed 19/50
Completed 20/50
Completed 21/50
Completed 22/50
Completed 23/50
Completed 24/50
Completed 25/50
Completed 26/50
Completed 27/50
Completed 28/50
Completed 29/50
Completed 30/50
Completed 31/50
Completed 32/50
Completed 33/50
Completed 34/50
Completed 35/50
Completed 36/50
Completed 37/50

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-safeguard-20b` in organization `org_01ksf8zjbdekv9j9pyrkvb9h3w` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7

In [28]:
completion_id = response.id
completion_id

'chatcmpl-86491806-6679-4925-a8cd-ba2b045b688c'

Reads directly from items

In [31]:
import json
import time
from litellm import completion

for item in items[:50]:

    while True:
        try:
            response = completion(
                model="groq/openai/gpt-oss-safeguard-20b",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": item.full}
                ]
            )

            item.summary = response.choices[0].message.content

            print(f"Completed item {item.id}")

            time.sleep(3)
            break

        except Exception as e:
            print(f"Error: {e}")
            time.sleep(10)

Completed item 0
Completed item 1
Completed item 2
Completed item 3
Completed item 4
Completed item 5
Completed item 6
Completed item 7
Completed item 8
Completed item 9
Completed item 10
Completed item 11
Completed item 12
Completed item 13
Completed item 14
Completed item 15
Completed item 16
Completed item 17
Completed item 18
Completed item 19
Completed item 20
Completed item 21
Completed item 22
Completed item 23
Completed item 24
Completed item 25
Completed item 26
Completed item 27
Completed item 28
Completed item 29
Completed item 30
Completed item 31
Completed item 32
Completed item 33
Completed item 34
Completed item 35
Completed item 36
Completed item 37
Completed item 38
Completed item 39
Completed item 40
Completed item 41
Completed item 42
Completed item 43
Completed item 44
Completed item 45
Completed item 46
Completed item 47
Completed item 48
Completed item 49


In [32]:
print(items[0].full)

Odyssey  Krom Utility/Record Case, Black
['Protect your rare and hard to find records used for your special 45 rpm vinyl sets or just for safe keeping in our Krom™ series  black utility/record cases. Each has two individual compartments that hold a total of 120 7" vinyl records, 60 in each section, and can also be used as a utility case for cables and other essentials. Features include chrome plated hardware, a removable lid, a fully foam-lined interior and a heavy-duty latch and spring-loaded handle. Individual K45120 cases can be stacked on each other for convenient storage. Also available in silver ||.']
['Butterfly latch and spring-loaded handle', 'Padlock latch holes', 'Holds 120 7" vinyl records, 60 per compartment', 'Foam-lined interior', 'Rubber feet and stacking lid']
{"Item Weight": "7 pounds", "Product Dimensions": "9.25 x 17.75 x 9.75 inches", "Is Discontinued By Manufacturer": "No", "Date First Available": "September 6, 2012", "Color Name": "BLACK", "Material Type": "Rubbe

In [37]:
print(items[49].full)

Ibanez 6 String Solid-Body Electric Guitar, Right, Blue ()
['GRG and GRX guitars feature the same playability, warranty, and set-up as our most expensive instruments, but at rock bottom affordable prices. The quilted art grain top and basswood body are available in three distinct transparent colors: blue burst, black sunburst, and red burst. Features: Slim & comfortable GRG maple necks Powerful Ibanez humbucker pickups Low profile designed Ibanez bridges for playing comfort Neck Material: Maple &; Neck Type: GRX &; Body: Basswood body/ Quilted Art Grain top &; Frets: Medium frets ; Fingerboard: Rosewood ; Inlay: Pearl dot inlay ; Bridge: FAT 6 bridge ; Neck PU: PSND1 ; Middle PU: PSND-S ; Bridge PU: PSND2 ; HW Color: CH ; Finish: Transparent Blue Burst']
['GRX Maple Neck', 'Poplar Body/ Quilted Art Grain Top', 'Medium frets', 'Rosewood Fingerboard', 'Pearl Dot Inlay']
{"Item Weight": "8 pounds", "Product Dimensions": "6 x 12 x 43 inches", "Country of Origin": "China", "Is Discontinued 

In [38]:
print(items[49].summary)

Title: Ibanez 6‑String Solid‑Body Electric Guitar – Right‑Handed Blue Burst  
Category: Musical Instruments  
Brand: Ibanez  
Description: A right‑handed, 6‑string solid‑body electric with a striking blue burst finish, maple neck, and powerful humbucker pickups.  
Details: The guitar features a basswood body with a quilted maple top, medium frets, rosewood fingerboard, pearl‑dot inlays, and a low‑profile FAT 6 bridge.


In [39]:
print(items[50].summary)

None


In [43]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

In [44]:
username = "SujithVarma2005"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:40]
    val = items[40:45]
    test = items[45:50]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)


Exception ignored in: <function tqdm.__del__ at 0x000002C6180E6480>
Traceback (most recent call last):
  File "c:\Users\nraja\OneDrive\Desktop\llama\llm_engineering\.venv\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "c:\Users\nraja\OneDrive\Desktop\llama\llm_engineering\.venv\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 42.86ba/s]
Processing Files (1 / 1): 100%|██████████| 20.1kB / 20.1kB, 1.88kB/s  

Processing Files (1 / 1): 100%|██████████| 20.1kB / 20.1kB, 1.88kB/s  
New Data Upload: 100%|██████████| 20.1kB / 20.1kB, 1.88kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 179.96ba/s]
Processing Files (1 / 1): 100%|██████████| 7.85kB / 7.85kB,   767B/s  
New Data Upload: 100%|██████████| 7.85kB / 7.85kB,   767B/s  
